In [2]:
# Core libraries
import numpy as np
import pandas as pd
import time

# Data generation & preprocessing
from sklearn.datasets import make_classification
from sklearn.model_selection import train_test_split, GridSearchCV, RandomizedSearchCV
from sklearn.preprocessing import StandardScaler

# Model & metrics
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, f1_score

# Reproducibility
RANDOM_STATE = 42

In [3]:
X, y = make_classification(
    n_samples=1000,
    n_features=20,
    n_informative=10,
    n_redundant=5,
    n_clusters_per_class=2,
    weights=[0.9, 0.1],   # 90% placed, 10% unplaced
    random_state=RANDOM_STATE
)

In [5]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    stratify=y,
    random_state=RANDOM_STATE
)

In [7]:
scaler = StandardScaler()

X_train_scaled = scaler.fit_transform(X_train)  # Fit only on training data
X_test_scaled = scaler.transform(X_test)

In [8]:
baseline_model = RandomForestClassifier(random_state=RANDOM_STATE)
baseline_model.fit(X_train_scaled, y_train)

y_pred_baseline = baseline_model.predict(X_test_scaled)

baseline_accuracy = accuracy_score(y_test, y_pred_baseline)
baseline_f1 = f1_score(y_test, y_pred_baseline)

print("Baseline Accuracy:", baseline_accuracy)
print("Baseline F1-Score:", baseline_f1)

Baseline Accuracy: 0.91
Baseline F1-Score: 0.3076923076923077


In [9]:
param_grid = {
    "n_estimators": [50, 100, 200],
    "max_depth": [None, 10, 20],
    "min_samples_split": [2, 5, 10]
}



In [11]:
grid_accuracy = GridSearchCV(
    estimator=RandomForestClassifier(random_state=RANDOM_STATE),
    param_grid=param_grid,
    scoring="accuracy",
    cv=5,
    n_jobs=-1
)

grid_accuracy.fit(X_train_scaled, y_train)

best_acc_model = grid_accuracy.best_estimator_

In [12]:
y_pred_acc = best_acc_model.predict(X_test_scaled)

acc_accuracy = accuracy_score(y_test, y_pred_acc)
acc_f1 = f1_score(y_test, y_pred_acc)

print("GridSearch (Accuracy) - Accuracy:", acc_accuracy)
print("GridSearch (Accuracy) - F1:", acc_f1)

GridSearch (Accuracy) - Accuracy: 0.92
GridSearch (Accuracy) - F1: 0.42857142857142855


In [13]:
grid_f1 = GridSearchCV(
    estimator=RandomForestClassifier(random_state=RANDOM_STATE),
    param_grid=param_grid,
    scoring="f1",
    cv=5,
    n_jobs=-1
)

grid_f1.fit(X_train_scaled, y_train)

best_f1_model = grid_f1.best_estimator_

In [14]:
y_pred_f1 = best_f1_model.predict(X_test_scaled)

f1_accuracy = accuracy_score(y_test, y_pred_f1)
f1_f1 = f1_score(y_test, y_pred_f1)

print("GridSearch (F1) - Accuracy:", f1_accuracy)
print("GridSearch (F1) - F1:", f1_f1)

GridSearch (F1) - Accuracy: 0.905
GridSearch (F1) - F1: 0.24


In [15]:
start_time = time.time()

grid_f1.fit(X_train_scaled, y_train)

grid_time = time.time() - start_time
best_grid_f1 = grid_f1.best_score_

In [16]:
random_param_dist = {
    "n_estimators": np.arange(10, 501),
    "max_depth": [None] + list(np.arange(5, 31)),
    "min_samples_split": np.arange(2, 11)
}

In [17]:
random_search = RandomizedSearchCV(
    estimator=RandomForestClassifier(random_state=RANDOM_STATE),
    param_distributions=random_param_dist,
    n_iter=20,
    scoring="f1",
    cv=5,
    random_state=RANDOM_STATE,
    n_jobs=-1
)

In [18]:
start_time = time.time()

random_search.fit(X_train_scaled, y_train)

random_time = time.time() - start_time
best_random_f1 = random_search.best_score_

In [19]:
comparison_df = pd.DataFrame({
    "Search Method": ["GridSearchCV", "RandomizedSearchCV"],
    "Time Taken (seconds)": [grid_time, random_time],
    "Best CV F1-Score": [best_grid_f1, best_random_f1]
})

comparison_df

,Search Method,Time Taken (seconds),Best CV F1-Score
0,GridSearchCV,5.574644,0.403463
1,RandomizedSearchCV,10.007516,0.385733
